# exp011 hyperparameter tuning v2

## 0. Experiment Metadata

In [4]:
# ========================================
# EXPERIMENT CONFIG
# ========================================

EXP_NAME = "exp011_hyperparameter_tuning_v2"
TARGET = "Churn"
ID_COL = "id"

N_SPLITS = 5
SEED = 42

print(f"Running {EXP_NAME}")

Running exp011_hyperparameter_tuning_v2


## 1. Imports

In [5]:
# ========================================
# IMPORTS
# ========================================

import os
import random
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

import lightgbm as lgb
import xgboost as xgb

import warnings
warnings.filterwarnings("ignore")

pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',50)

## 2. Reproducibility

In [6]:
# ========================================
# SEED EVERYTHING
# ========================================

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

seed_everything(SEED)

## 3. Load Data

In [7]:
# ========================================
# LOAD DATA
# ========================================

DATA_PATH = "/Users/theojeremiah/Workspace/01_DataScience/Projects/202603_Kaggle_CustomerChurn/data/raw/"

train = pd.read_csv(DATA_PATH + "train.csv")
test = pd.read_csv(DATA_PATH + "test.csv")

print(train.shape, test.shape)
train.head()

(594194, 21) (254655, 20)


,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,No,Yes,Yes,No,No,One year,Yes,Mailed check,60.10,1653.85,No
1,1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,Yes,No,Yes,Yes,No,Two year,No,Credit card (automatic),69.50,3778.20,No
2,2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,Yes,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.40,5841.35,No
3,3,Female,0,No,No,1,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,69.70,70.70,Yes
4,4,Female,0,No,No,1,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.45,70.45,Yes


## 4. Quick Sanity Checs (Fast EDA)

In [8]:
# ========================================
# QUICK EDA
# ========================================

print("Target distribution:")
print(train['Churn'].value_counts(normalize=True))

print("\nMissing values (top 10):")
print(train.isnull().mean().sort_values(ascending=False).head(10))

Target distribution:
Churn
No     0.774792
Yes    0.225208
Name: proportion, dtype: float64

Missing values (top 10):
id                  0.0
DeviceProtection    0.0
TotalCharges        0.0
MonthlyCharges      0.0
PaymentMethod       0.0
PaperlessBilling    0.0
Contract            0.0
StreamingMovies     0.0
StreamingTV         0.0
TechSupport         0.0
dtype: float64


## 5. Feature Engineering 

In [9]:
# ========================================
# FEATURE ENGINEERING (EXP006 - COMBINED BEST)
# ========================================

service_cols = [
    "OnlineSecurity", "OnlineBackup", "DeviceProtection", 
    "TechSupport", "StreamingTV", "StreamingMovies"
]

def create_features(df):
    df = df.copy()

    # ========================================
    # FINANCIAL (STRONG SIGNAL)
    # ========================================
    if all(col in df.columns for col in ["TotalCharges", "tenure"]):
        df["avg_charge"] = df["TotalCharges"] / (df["tenure"] + 1)

    if all(col in df.columns for col in ["TotalCharges", "MonthlyCharges", "tenure"]):
        df["charge_diff"] = df["TotalCharges"] - (df["MonthlyCharges"] * df["tenure"])

    # ========================================
    # SERVICE USAGE (STRONG SIGNAL)
    # ========================================
    if all(col in df.columns for col in service_cols):
        df["num_services"] = (df[service_cols] == "Yes").sum(axis=1)

    # ========================================
    # BUSINESS LOGIC (VERY STRONG)
    # ========================================
    if "PaymentMethod" in df.columns:
        df["payment_risk"] = (df["PaymentMethod"] == "Electronic check").astype(int)

    if "InternetService" in df.columns:
        df["is_fiber"] = (df["InternetService"] == "Fiber optic").astype(int)

    # ========================================
    # OPTIONAL CLEANUP (IMPORTANT DECISION)
    # ========================================
    # Drop TotalCharges ONLY if using avg_charge + charge_diff
    # if "TotalCharges" in df.columns:
    #     df = df.drop(columns=["TotalCharges"])

    return df


# Apply
train = create_features(train)
test = create_features(test)

## 6. Target Encoding

In [10]:
# ========================================
# TARGET ENCODING (SIMPLE VERSION)
# ========================================

from category_encoders import TargetEncoder

# Get categorical columns from TRAIN
categorical_cols = train.select_dtypes(include="object").columns.tolist()

# Remove target column
categorical_cols = [col for col in categorical_cols if col != TARGET]

# Initialize encoder
encoder = TargetEncoder(cols=categorical_cols)

# Fit on train, transform both
train[categorical_cols] = encoder.fit_transform(train[categorical_cols], train[TARGET])
test[categorical_cols] = encoder.transform(test[categorical_cols])

## 7. Prepare Data

In [11]:
# ========================================
# PREPARE DATA
# ========================================

features = [col for col in train.columns if col not in [TARGET, ID_COL]]

train[TARGET] = train[TARGET].map({"No": 0, "Yes": 1}).astype(int)

X = train[features]
y = train[TARGET]

X_test = test[features]

## 8. Validation Strategy

In [12]:
# ========================================
# CV SETUP
# ========================================

skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED
)

## 9. Model Training

### Optuna Light GBM

In [13]:
import optuna
import lightgbm as lgb
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective_lgb(trial):

    params = {
        "n_estimators": 2000,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.03),
        "num_leaves": trial.suggest_int("num_leaves", 31, 96),
        "max_depth": trial.suggest_int("max_depth", 4, 8),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 80),
        "subsample": trial.suggest_float("subsample", 0.7, 0.9),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 0.9),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 0.5),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 0.5),
        "random_state": 42,
        "n_jobs": 4,
        "verbosity": -1
    }

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    oof = np.zeros(len(X))

    for tr_idx, val_idx in skf.split(X, y):

        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

        model = lgb.LGBMClassifier(**params)

        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            callbacks=[
                lgb.early_stopping(100),
                lgb.log_evaluation(0)
            ]
        )

        oof[val_idx] = model.predict_proba(X_val)[:, 1]

    return roc_auc_score(y, oof)


study_lgb = optuna.create_study(direction="maximize")

study_lgb.optimize(
    objective_lgb,
    n_trials=50,
    show_progress_bar=True
)

BEST_LGB_PARAMS = study_lgb.best_params

  0%|          | 0/50 [00:00<?, ?it/s]

Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1632]	valid_0's binary_logloss: 0.298577
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1606]	valid_0's binary_logloss: 0.296514
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1758]	valid_0's binary_logloss: 0.297788
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1569]	valid_0's binary_logloss: 0.295907
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1809]	valid_0's binary_logloss: 0.300198
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's binary_logloss: 0.298226
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[1998]	valid_0's binary_logloss: 0.296355
T

### optuna XGBOOST

In [15]:
import xgboost as xgb

def objective_xgb(trial):

    params = {
        "n_estimators": 2000,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.03),
        "max_depth": trial.suggest_int("max_depth", 3, 6),
        "min_child_weight": trial.suggest_int("min_child_weight", 2, 8),
        "subsample": trial.suggest_float("subsample", 0.7, 0.9),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 0.9),
        "gamma": trial.suggest_float("gamma", 0.0, 2.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 0.5),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.5, 1.5),
        "eval_metric": "auc",
        "random_state": 42,
        "n_jobs": -1,
        "tree_method": "hist"
    }

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    oof = np.zeros(len(X))

    for tr_idx, val_idx in skf.split(X, y):

        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

        model = xgb.XGBClassifier(**params)

        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            verbose=False
        )

        oof[val_idx] = model.predict_proba(X_val)[:, 1]

    return roc_auc_score(y, oof)


study_xgb = optuna.create_study(direction="maximize")

study_xgb.optimize(
    objective_xgb,
    n_trials=50,
    show_progress_bar=True
)

BEST_XGB_PARAMS = study_xgb.best_params

  0%|          | 0/50 [00:00<?, ?it/s]

### use best params

In [16]:
BEST_LGB_PARAMS = study_lgb.best_params
BEST_XGB_PARAMS = study_xgb.best_params

In [17]:
BEST_XGB_PARAMS

{'learning_rate': 0.0271494620259345,
 'max_depth': 5,
 'min_child_weight': 5,
 'subsample': 0.8425072471777342,
 'colsample_bytree': 0.7021033916313842,
 'gamma': 0.8387184695096837,
 'reg_alpha': 0.07602576287100608,
 'reg_lambda': 1.0561196490793612}

### adjust based on the best param to run the model

In [18]:
# ========================================
# SEED EVERYTHING
# ========================================
def seed_everything(seed):
    import random, os
    import numpy as np

    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


# ========================================
# CONFIG
# ========================================
SEEDS = [42, 52, 62]
N_SPLITS = 5

oof_preds_lgb = np.zeros(len(train))
oof_preds_xgb = np.zeros(len(train))

test_preds_lgb = np.zeros(len(test))
test_preds_xgb = np.zeros(len(test))


# ========================================
# MULTI-SEED TRAINING
# ========================================
for seed in SEEDS:
    
    print(f"\n====================")
    print(f"SEED {seed}")
    print(f"====================")

    seed_everything(seed)

    skf = StratifiedKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=seed
    )

    oof_lgb_seed = np.zeros(len(train))
    oof_xgb_seed = np.zeros(len(train))

    test_lgb_seed = np.zeros(len(test))
    test_xgb_seed = np.zeros(len(test))

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        
        print(f"\n--- Fold {fold} ---")

        X_train, X_valid = X.iloc[tr_idx], X.iloc[val_idx]
        y_train, y_valid = y.iloc[tr_idx], y.iloc[val_idx]

        # ======================
        # LIGHTGBM
        # ======================
        lgb_model = lgb.LGBMClassifier(
            **BEST_LGB_PARAMS,
            random_state=seed,
            n_jobs=-1
        )

        lgb_model.fit(
            X_train, y_train,
            eval_set=[(X_valid, y_valid)],
            callbacks=[
                lgb.early_stopping(100),
                lgb.log_evaluation(200)
            ]
        )

        val_pred_lgb = lgb_model.predict_proba(X_valid)[:, 1]
        oof_lgb_seed[val_idx] = val_pred_lgb
        test_lgb_seed += lgb_model.predict_proba(X_test)[:, 1] / N_SPLITS


        # ======================
        # XGBOOST
        # ======================
        xgb_model = xgb.XGBClassifier(
            **BEST_XGB_PARAMS,
            n_jobs=-1
        )

        xgb_model.fit(
            X_train, y_train,
            eval_set=[(X_valid, y_valid)],
            verbose=False
        )

        val_pred_xgb = xgb_model.predict_proba(X_valid)[:, 1]
        oof_xgb_seed[val_idx] = val_pred_xgb
        test_xgb_seed += xgb_model.predict_proba(X_test)[:, 1] / N_SPLITS


    # ======================
    # ACCUMULATE RESULTS
    # ======================
    oof_preds_lgb += oof_lgb_seed / len(SEEDS)
    oof_preds_xgb += oof_xgb_seed / len(SEEDS)

    test_preds_lgb += test_lgb_seed / len(SEEDS)
    test_preds_xgb += test_xgb_seed / len(SEEDS)


# ========================================
# CV SCORES (INDIVIDUAL)
# ========================================
score_lgb = roc_auc_score(y, oof_preds_lgb)
score_xgb = roc_auc_score(y, oof_preds_xgb)

print("\nLGB CV:", score_lgb)
print("XGB CV:", score_xgb)


# ========================================
# ENSEMBLE WEIGHT OPTIMIZATION
# ========================================
best_score = 0
best_w = 0

for w in np.arange(0.0, 1.01, 0.01):
    oof_comb = w * oof_preds_lgb + (1 - w) * oof_preds_xgb
    score = roc_auc_score(y, oof_comb)

    if score > best_score:
        best_score = score
        best_w = w

print(f"\nBest Weight (LGB): {best_w:.2f}")
print(f"Best Ensemble CV: {best_score:.5f}")


# ========================================
# FINAL TEST PREDICTIONS
# ========================================
test_preds = best_w * test_preds_lgb + (1 - best_w) * test_preds_xgb


SEED 42

--- Fold 0 ---
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[100]	valid_0's binary_logloss: 0.313079

--- Fold 1 ---
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[100]	valid_0's binary_logloss: 0.31154

--- Fold 2 ---
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[100]	valid_0's binary_logloss: 0.312181

--- Fold 3 ---
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[100]	valid_0's binary_logloss: 0.310878

--- Fold 4 ---
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[100]	valid_0's binary_logloss: 0.314484

SEED 52

--- Fold 0 ---
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[100]	valid_0's binary

#### prev score 0.91669

## 11. Feature Importance

In [113]:
# ========================================
# FEATURE IMPORTANCE
# ========================================

import matplotlib.pyplot as plt

importance = pd.DataFrame({
    "feature": features,
    "importance": model.feature_importances_
}).sort_values(by="importance", ascending=False)

importance.head(20)

plt.figure(figsize=(8, 10))
plt.barh(importance["feature"].head(20), importance["importance"].head(20))
plt.gca().invert_yaxis()
plt.title("Top Features")
plt.show()

ValueError: All arrays must be of the same length

## 12. Create Submission

In [19]:
# ========================================
# SUBMISSION (PROBABILITY - CORRECT)
# ========================================

import os

# Define path
OUTPUT_DIR = "/Users/theojeremiah/Workspace/01_DataScience/Projects/202603_Kaggle_CustomerChurn/outputs/submissions/"

# Create folder if not exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Create submission (USE PROBABILITY)
submission = pd.DataFrame({
    ID_COL: test[ID_COL],
    TARGET: test_preds   # ✅ probability, NOT binary
})

# File path
file_path = os.path.join(OUTPUT_DIR, f"{EXP_NAME}.csv")

# Save
submission.to_csv(file_path, index=False)

print(f"Submission saved at: {file_path}")

Submission saved at: /Users/theojeremiah/Workspace/01_DataScience/Projects/202603_Kaggle_CustomerChurn/outputs/submissions/exp011_hyperparameter_tuning_v2.csv
